In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv("Nassau Candy Distributor.csv")
factories = pd.read_csv("factory_locations.csv")
product_factory = pd.read_csv("proposed_product_factory_mapping.csv")
print("Dataset loaded:", df.shape)
print("Factories:", len(factories))
print("Product mappings:", len(product_factory))

Dataset loaded: (10194, 18)
Factories: 5
Product mappings: 15


In [2]:
product_summary = (
    df.groupby("Product Name")
      .agg(
          Total_Sales=("Sales", "sum"),
          Total_Profit=("Gross Profit", "sum"),
          Total_Cost=("Cost", "sum"),
          Total_Units=("Units", "sum")
      )
      .reset_index()
)

product_summary["Profit_Margin_Percent"] = np.where(
    product_summary["Total_Sales"] > 0,
    product_summary["Total_Profit"]
    / product_summary["Total_Sales"] * 100,
    np.nan
)
display(product_summary.head())

,Product Name,Total_Sales,Total_Profit,Total_Cost,Total_Units,Profit_Margin_Percent
0,Everlasting Gobstopper,130.00,104.00,26.0,13,80.000000
1,Fizzy Lifting Drinks,78.75,47.25,31.5,21,60.000000
2,Fun Dip,12.00,4.80,7.2,8,40.000000
3,Hair Toffee,76.50,59.50,17.0,17,77.777778
4,Kazookles,1205.75,92.75,1113.0,371,7.692308


In [3]:
simulation_data = product_summary.merge(
    product_factory,
    on="Product Name",
    how="left",
    indicator=True
)

print("Products with a proposed factory:",
      (simulation_data["_merge"] == "both").sum())

print("Products without a proposed factory:",
      (simulation_data["_merge"] == "left_only").sum())

simulation_data.drop(columns="_merge", inplace=True)

display(simulation_data)

Products with a proposed factory: 14
Products without a proposed factory: 1


,Product Name,Total_Sales,Total_Profit,Total_Cost,Total_Units,Profit_Margin_Percent,Current Factory
0,Everlasting Gobstopper,130.00,104.00,26.00,13,80.000000,Secret Factory
1,Fizzy Lifting Drinks,78.75,47.25,31.50,21,60.000000,Sugar Shack
2,Fun Dip,12.00,4.80,7.20,8,40.000000,Sugar Shack
3,Hair Toffee,76.50,59.50,17.00,17,77.777778,The Other Factory
4,Kazookles,1205.75,92.75,1113.00,371,7.692308,The Other Factory
5,Laffy Taffy,53.73,33.48,20.25,27,62.311558,Sugar Shack
6,Lickable Wallpaper,7860.00,3930.00,3930.00,393,50.000000,Secret Factory
7,Nerds,15.00,7.00,8.00,10,46.666667,Sugar Shack
8,SweeTARTS,61.50,28.70,32.80,41,46.666667,Sugar Shack
9,Wonka Bar - Fudge Mallows,24890.40,16593.60,8296.80,6914,66.666667,Lot's O' Nuts


In [4]:
simulation_data = simulation_data.sort_values(
    "Total_Profit",
    ascending=False
)

display(
    simulation_data[
        [
            "Product Name",
            "Current Factory",
            "Total_Sales",
            "Total_Profit",
            "Total_Cost",
            "Total_Units",
            "Profit_Margin_Percent"
        ]
    ].head(10)
)


,Product Name,Current Factory,Total_Sales,Total_Profit,Total_Cost,Total_Units,Profit_Margin_Percent
13,Wonka Bar -Scrumdiddlyumptious,NaN,27874.80,19357.50,8517.30,7743,69.444444
12,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,28485.00,18610.20,9874.80,7596,65.333333
10,Wonka Bar - Milk Chocolate,Wicked Choccy's,26867.75,17443.37,9424.38,8267,64.923077
11,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts,23574.95,16819.95,6755.00,6755,71.346705
9,Wonka Bar - Fudge Mallows,Lot's O' Nuts,24890.40,16593.60,8296.80,6914,66.666667
6,Lickable Wallpaper,Secret Factory,7860.00,3930.00,3930.00,393,50.000000
14,Wonka Gum,Secret Factory,597.50,310.70,286.80,478,52.000000
0,Everlasting Gobstopper,Secret Factory,130.00,104.00,26.00,13,80.000000
4,Kazookles,The Other Factory,1205.75,92.75,1113.00,371,7.692308
3,Hair Toffee,The Other Factory,76.50,59.50,17.00,17,77.777778


In [5]:
simulation_data.to_csv(
    "factory_reallocation_simulation_input.csv",
    index=False
)

print("Simulation input saved successfully!")


Simulation input saved successfully!
